In [ ]:
import pandas as pd
pd.set_option('display.max_columns', None)

import numpy as np
import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False
plt.style.use('fivethirtyeight')
import seaborn as sns 

### 0. 함수 정의
- **preprocessing** : 전처리 함수
- **check_relationship** : 한 테이블 내, 두 컬럼 간의 관계 확인
- **box_plot** : 수치형 컬럼 추출 후 시각화 진행

In [ ]:
def preprocessing(df, name="데이터프레임"):
    print(f"\n{'='*30} [{name}] 탐색 리포트 {'='*30}")
    
    # 1. 기본 정보 및 컬럼 타입 확인
    print("\n[📋 1. 기본 정보 및 데이터 타입]")
    df.info() 
    
    # 2. 결측치 상세 확인
    null_info = df.isnull().sum()
    if null_info.sum() > 0:
        print("\n[🚫 2. 결측치 상세 현황]")
        # 결측치가 있는 컬럼만 골라내어 개수와 비율을 계산
        null_df = pd.DataFrame({
            '결측치 수': null_info[null_info > 0],
            '비율 (%)': (null_info[null_info > 0] / len(df) * 100).round(2)
        })
        print(null_df)
    else:
        print("\n[🚫 2. 결측치 상세 현황]: 결측치가 없습니다! ✨")

    # 3. 중복값 및 PK(기본키) 후보 확인
    print("\n[🆔 3. 중복값 및 PK 후보 체크]")
    unique_found = False
    for col in df.columns:
        unique_count = df[col].nunique()
        
        # 모든 행의 값이 고유하다면 PK 후보로 간주
        if unique_count == len(df):
            print(f" 🌟 '{col}': 중복 없음 (PK 후보)")
            unique_found = True
        # 중복이 있지만 그 비율이 10% 미만인 경우 (실수나 특이 케이스 확인용)
        elif df[col].duplicated().any():
            dup_count = df[col].duplicated().sum()
            dup_ratio = (dup_count / len(df)) * 100
            if dup_ratio < 10:
                print(f" ⚠️ '{col}': 중복 데이터 {dup_count}건 감지 ({dup_ratio:.2f}%)")
    
    if not unique_found:
        print(" ℹ️ 모든 컬럼이 중복값을 가집니다.")

    # 4. 데이터 타입별 분류 요약
    print("\n[📊 4. 데이터 타입별 분류 요약]")
    # 수치형 데이터와 범주형(날짜 포함) 데이터를 분리하여 리스트업
    num_cols = df.select_dtypes(include=['number']).columns.tolist()
    obj_cols = df.select_dtypes(include=['object', 'category', 'datetime']).columns.tolist()
    
    print(f"- 수치형 컬럼 ({len(num_cols)}개): {num_cols}")
    print(f"- 범주형/기타 컬럼 ({len(obj_cols)}개): {obj_cols}")

    print(f"\n{'='*75}\n")

In [ ]:
def check_relationship(df, col_a, col_b):
    print(f"\n{'='*20} [{col_a}] vs [{col_b}] 관계 분석 {'='*20}")
    
    # A 하나당 B가 몇 개 있는지 확인
    a_to_b = df.groupby(col_a)[col_b].nunique().max()
    # B 하나당 A가 몇 개 있는지 확인
    b_to_a = df.groupby(col_b)[col_a].nunique().max()
    
    print(f"✔️ '{col_a}' 하나당 '{col_b}' 최대 개수: {a_to_b}")
    print(f"✔️ '{col_b}' 하나당 '{col_a}' 최대 개수: {b_to_a}")
    
    if a_to_b <= 1 and b_to_a <= 1:
        relation = "1 : 1"
    elif a_to_b > 1 and b_to_a <= 1:
        relation = "1 : N"
    elif a_to_b <= 1 and b_to_a > 1:
        relation = "N : 1"
    else:
        relation = "N : M (다대다)"
        
    print(f"\n📢 최종 관계: [ {relation} ]")
    
    # 1:N이나 N:M일 경우 예시 데이터 보여주기
    if a_to_b > 1:
        sample_a = df.groupby(col_a)[col_b].nunique().idxmax()
        print(f"\n💡 [1:N 예시] '{col_a}'의 {sample_a} 값에 연결된 '{col_b}'들:")
        print(df[df[col_a] == sample_a][col_b].unique())

    print(f"{'='*60}\n")

In [ ]:
def box_plot(df, name="DataFrame", max_unique=10):
    # 1. 수치형 컬럼 추출
    num_cols = df.select_dtypes(include=['number']).columns.tolist()
    
    # 2. 진짜 연속형 데이터에 가까운 것만 필터링
    # 고유값이 너무 적은 건 제외하거나 따로 처리
    target_cols = [col for col in num_cols if df[col].nunique() > max_unique]
    
    if not target_cols:
        print(f"[{name}] 시각화할 연속형 수치 데이터가 없습니다. (고유값 {max_unique}개 이하 제외)")
        return

    for col in target_cols:
        plt.figure(figsize=(12, 4))
        sns.violinplot(x=df[col], inner=None, color=".9")
        sns.boxplot(x=df[col], width=0.3, color="skyblue")
        plt.title(f'[{name}] {col} 분포 (고유값: {df[col].nunique()}개)', fontsize=14)
        plt.show()

In [ ]:
file_path ='C:/Users/82109/Downloads/olist_dataset/'
df_customers = pd.read_csv(file_path + 'olist_customers_dataset.csv')
df_geolocation = pd.read_csv(file_path + 'olist_geolocation_dataset.csv')
df_orders = pd.read_csv(file_path + 'olist_orders_dataset.csv')
df_items = pd.read_csv(file_path + 'olist_order_items_dataset.csv')
df_payments = pd.read_csv(file_path + 'olist_order_payments_dataset.csv')
df_reviews = pd.read_csv(file_path + 'olist_order_reviews_dataset.csv')
df_products = pd.read_csv(file_path + 'olist_products_dataset.csv')
df_sellers = pd.read_csv(file_path + 'olist_sellers_dataset.csv')

### 1. 주문 데이터
#### 현황 분석

- **1:1 관계**
- created (주문 생성) ➡️ approved (승인) ➡️ invoiced (송장 발행) ➡️ processing (출고 처리 중) ➡️ shipped (배송 시작) ➡️ delivered (배송 완료)
- 취소된 배송:canceled, unavailable
- 지표 생성:  
    - early_or_late_days: 약속된 배송일과 실제 배송일 격차 
    - is_delayed: 지연 여부를 확인
- 주문 기간: 2016-09-04 ~ 2018-10-17 (약 2년)
- 배송 기간: 중앙값 기준 11일 이며, 전체 주문의 50%는 6~16일 내 배송됨
- 배송 지연: 전반적인 배송은 예정일보다 일찍 도착하지만, 극단적인 조기/지연 배송이 섞여있음
    - 현황: 93.42%가 정상 배송, 6.57%가 배송 지연  

#### 다음 분석 방향
- 배송의 지연이 고객 경험(리뷰, 재구매)에 미치는 영향 분석 필요

In [ ]:
preprocessing(df_orders, "주문 데이터")

In [ ]:
# 분포 확인
print(df_orders.order_status.value_counts())

In [ ]:
#날짜형 변환
date_cols = [
    'order_purchase_timestamp', 'order_approved_at', 
    'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date'
]
for col in date_cols:
    df_orders[col] = pd.to_datetime(df_orders[col])

# early_or_late_days(약속일-실제도착일) 및 지연 여부 플래그 생성 
df_orders['early_or_late_days'] = (
    df_orders['order_estimated_delivery_date'].dt.normalize() - 
    df_orders['order_delivered_customer_date'].dt.normalize()
).dt.days

# 지연 여부 판단 (음수일 때 1, 정시/조기 도착 시 0)
df_orders['is_delayed'] = np.where(
    df_orders['early_or_late_days'].isna(), 
    np.nan, 
    (df_orders['early_or_late_days'] < 0).astype(float)
)
# 배송일 오류 데이터 처리 (도착일이 주문일보다 빠른 비정상 데이터 정제)
invalid_indices = df_orders[df_orders['order_delivered_customer_date'] < df_orders['order_purchase_timestamp']].index
df_orders.loc[invalid_indices, ['early_or_late_days', 'is_delayed']] = np.nan

In [ ]:
#분포 꼬리 확인
print(df_orders['early_or_late_days'].describe())
print(df_orders['early_or_late_days'].quantile([0.01, 0.99]))

#배송 지연 분포
print(df_orders['is_delayed'].value_counts())
print(df_orders['is_delayed'].value_counts(normalize=True))

#샘플 확인
worst_5 = df_orders.nsmallest(5, 'early_or_late_days')[
    ['order_estimated_delivery_date', 'order_delivered_customer_date', 'early_or_late_days', 'is_delayed']
]
print("\n-----최대 지연 사례 (Top 5):-----")
print(worst_5)

# 배송완료일이 주문일보다 앞서는 경우가 있는지 확인
invalid_cases = df_orders[df_orders['order_delivered_customer_date'] < df_orders['order_purchase_timestamp']]
print(f"\n-----배송일 오류 데이터 건수: {len(invalid_cases)}-----")

In [ ]:
plt.figure(figsize=(12, 4))
sns.boxplot(x=df_orders['early_or_late_days'], width=0.5)
plt.axvline(x=0, color='red', linestyle='--',linewidth=2)
plt.axvspan(df_orders['early_or_late_days'].min(), 0, color='red', alpha=0.1)
plt.axvspan(0, df_orders['early_or_late_days'].max(), color='green', alpha=0.1)

#텍스트 추가 (188일 지점 강조)
min_val = df_orders['early_or_late_days'].min()
plt.annotate(f'최대 지연: {abs(min_val)}일', 
             xy=(min_val, 0), xytext=(min_val + 20, -0.3),
             arrowprops=dict(facecolor='black', shrink=0.05, width=1, headwidth=5))

plt.xlabel('지연/조기 도착 일수 (음수 = 지연)', fontsize=15)
plt.tight_layout()
plt.show()

In [ ]:
#전체 배송 지연율과 지연 건수 
order_count = df_orders['order_id'].nunique()
total_latedelivery_count = df_orders['is_delayed'].sum()
delay_rate = round((total_latedelivery_count / order_count)*100, 2)

print(f'전체 배송 지연율: {delay_rate}%')
print(f'전체 배송 지연 건수: {total_latedelivery_count}')

In [ ]:
df_delayed = df_orders[df_orders['early_or_late_days'] < 0].copy()
df_delayed['delay_days'] = df_delayed['early_or_late_days'].abs()

bins = [0, 3, 6, 10, 20, np.inf]
labels = ['1-3', '4-6', '7-10', '11-20', '21~']

df_delayed['delay_bins'] = pd.cut(df_delayed['delay_days'], bins=bins, labels=labels)

result = df_delayed.groupby('delay_bins', observed=True)['order_id'].nunique().reset_index()
result.columns = ['delay_bins', 'counts']
total_delayed_count = result['counts'].sum()
result['ratio'] = result['counts'] / total_delayed_count * 100

print(result)

In [ ]:
plt.figure(figsize=(10, 4))

# 막대 그래프 생성
bars = plt.bar(result['delay_bins'],result['counts']
               )
plt.title('지연 주문 중, 지연 일수 분포', fontsize=15)
for bar in bars:
    height = bar.get_height()
    ratio = (height / total_delayed_count) * 100
    
    label_text = f'{height}건\n({ratio:.1f}%)'
    plt.text(bar.get_x() + bar.get_width()/2, height, label_text,
             ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.ylim(0, result['counts'].max() * 1.15)
plt.show()


### 2. 결제 데이터
#### 현황 분석
- **N:M관계**, payment_sequential에 따라 데이터가 중복됨
- 결제 수단 중 신용카드가 74% 사용되고, 가장 고액의 금액이 결제됨(금액별 voleto > debit_card > voucher)
- 결제 금액 분포는 평균(154)과 중앙값(100) 차이가 크다(분포가 한쪽으로 치우쳐있다)
- **특이사항**: 결제 금액이 0원인 건은 'not_defined' 결제 수단으로 분류됨을 확인, 이는 오류가 아닌 특정 프로모션 케이스로 판단하여 분석 대상에 포함함

In [ ]:
preprocessing(df_payments, name="결제데이터")

In [ ]:
check_relationship(df_payments, 'order_id', 'payment_sequential')

In [ ]:
#결제금액 통계 및 분포
print('-----결제금액 통계 및 분포-----')
print(df_payments.payment_value.describe())

df_payments['payment_value'].quantile([0.01, 0.90,0.95, 0.99])

In [ ]:
box_plot(df_payments, name="결제 데이터", max_unique=100)

In [ ]:
#결제 수단별 결제 금액 비교
##신용카드가 가장 높은 결제 금액을 가짐
print('--결제수단별 결제 금액 평균 값--')
print(df_payments.groupby('payment_type')['payment_value'].mean().sort_values(ascending=False))
print('--결제수단별 결제 금액 중앙 값--')
print(df_payments.groupby('payment_type')['payment_value'].median().sort_values(ascending=False))

In [ ]:
value_per_type = df_payments.groupby('payment_type')['payment_value'].mean().sort_values(ascending=False)
sns.barplot(x=value_per_type.values, y=value_per_type.index, palette='Set2')
plt.title('결제 수단별 평균 결제 금액')
plt.xlabel('평균 금액')
plt.ylabel('결제 수단')
plt.show()

### 3. 고객 데이터
#### 현황 분석
- **1:N관계** customer_unique_id는 고객 단위, customer_id는 주문 단위로 주문에 따라 같은 고객마다 다른 customer_id가 생성 
 
- 1회 구매자가 약 96.95%로 대부분을 차지하고, 재구매율은 3.04%로 극히 낮음
- 주:상위 10개의 state에서 90%의 고객이 거주함
- 도시:sao paulo 등 소수 대도시에 고객이 집중됨 (sao paulo:15.6%, 상위 5개 도시:약 29%, SP주: 42%)

#### 다음 분석 방향
- 지역별 배송 기간 차이와 배송 지연 현황 분석 예정

In [ ]:
preprocessing(df_customers, name="고객 데이터")

In [ ]:
check_relationship(df_customers, 'customer_id', 'customer_unique_id')

In [ ]:
#지역별 분포
print('고객 거주하는 도시의 수: ', df_customers.customer_city.nunique())
print('고객 거주하는 주의 수: ', df_customers.customer_state.nunique())
print(df_customers['customer_city'].value_counts(normalize=True).head())
print(df_customers['customer_state'].value_counts(normalize=True).head())

In [ ]:
top_states = df_customers['customer_state'].value_counts().head(10)
print(top_states)

buttom_states =  df_customers['customer_state'].value_counts().tail(10).sort_values()
print(buttom_states)

In [ ]:
#상위 10개의 state에서 90%의 고객이 거주함
state_ratio = df_customers['customer_state'].value_counts(normalize=True)
print(state_ratio.head())
print(state_ratio.cumsum())

In [ ]:
city_ratio = df_customers['customer_city'].value_counts(normalize=True)
print(city_ratio.head())
print(city_ratio.cumsum().head(20))

In [ ]:
plt.figure(figsize=(12,6))
ax = sns.barplot(x= top_states.index, y=top_states.values, palette ='Set3')
for i,v in enumerate(top_states.values):
    ax.text(i, v+150, f'{v:,}', ha='center', fontweight='bold', fontsize=10)

sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
#전체 재구매율 
df_orders_customers = pd.merge(df_orders,df_customers[['customer_id','customer_unique_id']], on='customer_id')
df_clean = df_orders_customers[~df_orders_customers['order_status'].isin(['canceled', 'unavailable'])].copy()

user_order_counts = df_clean.groupby('customer_unique_id')['order_id'].nunique().reset_index()
user_order_counts.columns = ['customer_unique_id', 'order_count']

#재구매여부 판별
user_order_counts['is_repurchase'] = (user_order_counts['order_count'] > 1).astype(int)

#전체 재구매율 계산
total_users = len(user_order_counts)
repurchase_users = user_order_counts['is_repurchase'].sum()
total_repurchase_rate = (repurchase_users / total_users) * 100

print(f"전체 재구매율: {total_repurchase_rate:.2f}%")

In [ ]:
#구매 횟수 분포
print(user_order_counts['order_count'].value_counts())
print(user_order_counts['order_count'].value_counts(normalize=True))

### 4 주문 아이템 데이터
#### 현황 분석

- **1:N관계** 주문한 아이템의 갯수에 따라 상세 정보가 나타남

- 상품 가격 분포를 보면, 대부분의 상품은 저~중가에 몰려있음
    - 평균 가격 약 120.65헤알, 중앙값은 약 74.99헤알
- 배송비 분포도 긴 우측 꼬리 가짐
    - 평균은 약 19.99헤알, 중앙값은 16.26헤알
    
- 전체의 약 97% 고객이 한 주문 당 3개 이하로 구매 (전체의 87%의 고객이 1개 구매)
- 셀러 처리 지연은 약 9%로 10413건 존재

#### 앞으로 분석 방향

- 셀러의 처리 지연이 전체 배송 지연에 미치는 영향을 분석할 필요가 있음

In [ ]:
preprocessing(df_items, name="아이템 데이터")

In [ ]:
check_relationship(df_items, 'order_id','order_item_id')

In [ ]:
# 주문당 구매 아이템 갯수 분포
print(df_items['order_item_id'].value_counts().head())
print(df_items['order_item_id'].value_counts(normalize=True).head())
print(df_items['order_item_id'].value_counts(normalize=True).cumsum().head())

In [ ]:
#날짜형 변환
df_items['shipping_limit_date']= pd.to_datetime(df_items['shipping_limit_date'])

In [ ]:
#취소값 제외
df_orders_f = df_orders[~df_orders['order_status'].isin(['canceled', 'unavailable'])].copy()

#병합
df_orders_items = pd.merge(df_orders_f,df_items[['order_id', 'shipping_limit_date']], on='order_id')

#지연여부
df_orders_items['late_process'] = (df_orders_items['order_delivered_carrier_date'] > df_orders_items['shipping_limit_date']).map({
    True: '지연', False: '정상'
})
print(df_orders_items['late_process'].value_counts())
print(df_orders_items['late_process'].value_counts(normalize=True))

In [ ]:
#월별 지연율 수치, 증감률, 추이
df_orders['month'] = df_orders['order_purchase_timestamp'].dt.to_period('M')
df_orders_items= df_orders.merge(df_items, on='order_id')
df_orders_items['is_seller_delayed'] = df_orders_items['order_delivered_carrier_date'].dt.normalize() > df_orders_items['shipping_limit_date'].dt.normalize()
df_orders_items['is_long_delay'] = df_orders_items['early_or_late_days'] <= -6

df_order_level = df_orders_items.groupby('order_id').agg({
    'month': 'first',
    'is_delayed': 'max',        
    'is_seller_delayed': 'max',   
    'is_long_delay': 'max'    
}).reset_index()

# 월별 그룹화 후 집계
monthly = df_order_level.groupby('month').agg(
    total_orders=('order_id', 'nunique'),
    delayed_orders=('is_delayed', 'sum'),
    seller_delayed_orders=('is_seller_delayed', 'sum'),
    long_delay_orders=('is_long_delay', 'sum')
)

# 비율 계산
monthly['delay_rate'] = monthly['delayed_orders'] / monthly['total_orders']
monthly['seller_delay_rate'] = monthly['seller_delayed_orders'] / monthly['total_orders']
monthly['long_days_delay_rate'] = monthly['long_delay_orders'] / monthly['delayed_orders']

In [ ]:
# shift를 사용하여 전월 데이터 가져오기
monthly['prev_delay_rate'] = monthly['delay_rate'].shift(1)
monthly['prev_seller_rate'] = monthly['seller_delay_rate'].shift(1)
monthly['prev_long_days_delay_rate'] = monthly['long_days_delay_rate'].shift(1)

# MoM 계산
monthly['delay_rate_mom'] = (monthly['delay_rate'] - monthly['prev_delay_rate']) / monthly['prev_delay_rate']
monthly['seller_delay_rate_mom'] = (monthly['seller_delay_rate'] - monthly['prev_seller_rate']) / monthly['prev_seller_rate']
monthly['long_days_delay_rate_mom'] = (monthly['long_days_delay_rate'] - monthly['prev_long_days_delay_rate']) / monthly['prev_long_days_delay_rate']

result = monthly[['delay_rate', 'delay_rate_mom', 'seller_delay_rate', 'seller_delay_rate_mom', 'long_days_delay_rate', 'long_days_delay_rate_mom']].round(4)
monthly

In [ ]:
plt.figure(figsize=(12, 5))
target_monthly = monthly.loc['2016-10':'2018-08']

colors = sns.color_palette('Set2')
sns.lineplot(data=target_monthly, x=target_monthly.index.astype(str), y='delay_rate', color=colors[0], label='배송 지연율')
sns.lineplot(data=target_monthly, x=target_monthly.index.astype(str), y='seller_delay_rate', color=colors[1], label='셀러 발송 지연율')
sns.lineplot(data=target_monthly, x=target_monthly.index.astype(str), y='long_days_delay_rate', color=colors[2], label='6일 이상 배송 지연율')

plt.title('월별 지연 추이', fontsize=15)
plt.xlabel('Month', fontsize=12)
plt.xticks(rotation=45)
plt.legend()
plt.grid(True, linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()

In [ ]:
#가격과 배송비 분포
print(df_items['price'].describe())
print(df_items['freight_value'].describe())

In [ ]:
box_plot(df_items, name="아이템 데이터", max_unique=100)

### 5. 리뷰 데이터
#### 현황 분석
 
- 주문 아이디와 리뷰 아이디는 **다대다 관계(N:M)**
    - **1. 하나의 주문 ID - 여러개의 리뷰 ID**
        - 점수 변동 주문: 202건
        - 내용 변동 주문: 126건
        - 날짜 변동 주문: 547건

    - **2. 여러 주문 ID - 하나의 리뷰 ID** 
- 리뷰 점수는 전반적으로 높으나, 불만족 고객(1점)이 중간 만족(2~3점) 고객보다 더 많이 존재하는 양극화된 분포를 보임
- **대부분의 고객이 리뷰 스코어를 제출함**
    - 리뷰 쓴 비율: 98.96% / 리뷰 안쓴 사람 수: 1031

#### 다음 분석 방향
- 배송의 지연 상태가 리뷰점수에 어떤 영향을 주는지 분석이 필요함

In [ ]:
preprocessing(df_reviews, name="리뷰 데이터")

In [ ]:
check_relationship(df_reviews, 'review_id', 'order_id')

In [ ]:
#하나의 리뷰에 여러개 주문 아이디를 가질 때

id_num_check2 = df_reviews.groupby('review_id')['order_id'].nunique()
multi_order = id_num_check2.loc[id_num_check2>1].index
df_multi_order = df_reviews[df_reviews['review_id'].isin(multi_order)].sort_values(by='review_id')
df_multi_order.head(20)

In [ ]:
df_orders_reviews= df_orders.merge(df_reviews, on='order_id')
not_review = (df_orders_reviews['order_id'].nunique() - df_orders_reviews['review_id'].nunique())
review_pct = df_orders_reviews['review_id'].nunique() / df_orders_reviews['order_id'].nunique()
print(f'리뷰 쓴 비율: {round(review_pct,4)}')
print(f'리뷰 안쓴 사람 수: {not_review}')

In [ ]:
#리뷰 스코어 분포
df_reviews_f = df_reviews.sort_values('review_answer_timestamp').drop_duplicates('order_id', keep='last')

print(df_reviews_f['review_score'].value_counts())
print(df_reviews_f['review_score'].value_counts(normalize=True))

In [ ]:
# 하나의 주문 ID - 여러개의 리뷰 ID
# 점수 그대로: 보강하거나, 리뷰를 더 구체화 / 점수 변화: 실제 사용 후 경험 변화

review_per_order = df_reviews.groupby('order_id')['review_id'].nunique()
multi_review_order = review_per_order[review_per_order > 1].index
df_multi_reviews = df_reviews[df_reviews['order_id'].isin(multi_review_order)].sort_values(['order_id', 'review_answer_timestamp'])

review_diff = df_multi_reviews.groupby('order_id')['review_score'].agg(['first', 'last'])

conditions = [
review_diff['last'] > review_diff['first'],
review_diff['last'] < review_diff['first']
]

choices = ['Increased', 'Decreased']
review_diff['change_type'] = np.select(conditions, choices, default='No Change')
review_diff['change_type'].value_counts()

In [ ]:
plt.figure(figsize=(10,4))
ax = sns.countplot(data=review_diff, x='change_type', palette = 'Set2')
for p in ax.patches:
    ax.annotate(f'{int(p.get_height())}건', 
                (p.get_x() + p.get_width() / 2., p.get_height()), 
                ha = 'center', va = 'center', 
                xytext = (0, 9), 
                textcoords = 'offset points',
                fontsize=12, fontweight='bold')


plt.title('리뷰 재작성 시 점수 변화 분포',fontsize=16, pad = 20)
plt.xlabel('변화 유형', fontsize= 12)
plt.ylabel('주문 건수', fontsize= 12)
plt.ylim(0,400)
plt.show()

### 6. 제품 데이터
#### 현황 분석
- 총 73개의 카테고리, 32951개의 제품이 있음

In [ ]:
preprocessing(df_products, name="제품 데이터")

In [ ]:
#카테고리 갯수
print(f"카테고리 갯수: {df_products['product_category_name'].nunique()}")
#카테고리별 상품 분포
print(df_products['product_category_name'].value_counts().head(10))

In [ ]:
#카테고리별 매출 순위
df_items_products = df_items.merge(
                    df_products[['product_id', 'product_category_name']], 
                    on = 'product_id'
                    )

df_items_products['revenue']= df_items_products['price'] + df_items_products['freight_value']                                                
category_revenue = df_items_products.groupby('product_category_name')['revenue'].sum().sort_values(ascending=False)
category_revenue.head(20)

### 7. 셀러 데이터
#### 현황 분석
- 3095명의 셀러 존재
- 셀러는 도시 기준 sao paulo에 약 22%가 집중되어 있으며,
- 주(state) 기준으로는 SP가 전체의 약 60%를 차지해 셀러가 특정 지역에 편중된 구조임을 확인

In [ ]:
preprocessing(df_sellers, name="셀러 데이터")

In [ ]:
#셀러 지역 분포
print(df_sellers['seller_city'].value_counts().head())
print(df_sellers['seller_city'].value_counts(normalize=True).head())
print(df_sellers['seller_state'].value_counts().head())
print(df_sellers['seller_state'].value_counts(normalize=True).head())

In [ ]:
sellers_items = df_sellers.merge(df_items, on='seller_id')
sellers_items_orders = sellers_items.merge(df_orders, on = 'order_id')
sellers_items_orders

In [ ]:
seller_order_count = sellers_items_orders.groupby('seller_id')['order_id'].nunique().reset_index()
seller_order_count['order_id'].describe()


In [ ]:
# 평균과 중앙값 산출
avg_order_count = seller_order_count['order_id'].mean()
median_order_count = seller_order_count['order_id'].median()

print(f"셀러별 평균 주문 건수: {avg_order_count:.2f}")
print(f"셀러별 주문 건수 중앙값: {median_order_count:.2f}")

# 상세 분포 확인 (평균, 중앙값 포함)
print(seller_order_count['order_id'].describe())

In [ ]:
# 저장할 데이터프레임 리스트
dataframes = {
    'customers': df_customers,
    'geolocation': df_geolocation,
    'orders': df_orders,
    'items': df_items,
    'payments': df_payments,
    'reviews': df_reviews,
    'products': df_products,
    'sellers': df_sellers
}

# 하나씩 Pickle로 저장
for name, dframe in dataframes.items():
    dframe.to_pickle(f'cleaned_{name}.pkl')
    print(f'cleaned_{name}.pkl 저장 완료!')